# Riproduzione Bahmani et al. (2012) — Tabelle 3/4, Figure 5.1/5.2

Driver: `src/paper_experiments.py`. Piano e decisioni: `docs/ANALYSIS_PLAN.md`.

**Workflow**: questo notebook gira sul **VM head** (jupyter via tunnel
`localhost:4444`). Prima di eseguire: `git pull` del branch
`repo-reorganization`. Le sezioni marcate **[CLUSTER]** richiedono il cluster
avviato (`launch_cluster`); la sezione **Fig 5.2** gira anche in locale.

Stato default dei flag: tutto spento (`RUN_FIG52 = False`, ecc.) — accendere
una sezione alla volta.

## 0. Sanity check del repository

In [1]:
!git branch --show-current && git status --short && git log --oneline -3

repo-reorganization
 M analysis.ipynb
 M paper_reproduction.ipynb
 M run.ipynb
?? ../figures/worker_sweep_20260827115917_10pc.png
?? ../figures/worker_sweep_20260828095433_10pc.png
3ecabc1 (HEAD -> repo-reorganization, origin/repo-reorganization) Proceeded work; now trying to send updated kmeans_parallel.py to all workers in order to make cluster work with 4-5 workers only on full dataset
c52fcf2 Proceeded with code testing, suspect bug in handling partitions
423c5d7 added tracking


## 1. Import e flag

In [2]:
import os, time
import numpy as np
import pandas as pd

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset, make_gauss_mixture, array_to_bag
from src.paper_experiments import (
    run_fig51, run_fig52, run_table34,
    plot_fig51, plot_fig52, table34_cost_table, table34_time_table,
)
from src.benchmark import RESULTS_DIR

# --- flag di esecuzione: accendere UNA sezione alla volta ---
RUN_FIG52_TINY   = True   # griglia ridotta, locale (~1 min)
RUN_FIG52_FULL   = False   # griglia completa dell'articolo, locale (~ore)
RUN_CLUSTER      = True   # abilita le sezioni [CLUSTER]
RUN_FIG51        = True   # KDD 10%, exact-l (cluster, costo moderato)
RUN_TABLE_SANITY = True   # singola run k=500 l/k=10 PRIMA della sweep piena
RUN_TABLE34      = True   # KDD full (cluster, NOTTATA)

SEED = 42
N_RUNS = 11                # convenzione del paper: mediana su 11 run

## 2. Fig 5.2 — GaussMixture **[LOCALE]**

Griglia ridotta di validazione prima della corsa completa.

In [3]:
if RUN_FIG52_TINY:
    df52 = run_fig52(client=None,
                     R_values=(1,), l_over_k_values=(1.0, 2.0),
                     r_values=(0, 1, 2, 3), k=20, n=2_000, d=8,
                     n_runs=2, seed=SEED, max_iter_fit=30)
    df52.to_csv(os.path.join(RESULTS_DIR, "fig52_tiny.csv"), index=False)
    display(df52.groupby(["method", "l_over_k", "r"])["cost_final"].median())
else:
    print("RUN_FIG52_TINY = False")

[fig52] R=1: riferimento k-means++, 2 run
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8), 0.0 MB | state: (250, 2), 0.0 MB | new_centroids: (21, 8), 0.0 MB
M: (250, 8

method    l_over_k  r  
kmeans||  1.0       0.0    14194.672424
                    1.0    14102.990022
                    2.0    14139.888756
                    3.0    14232.421254
          2.0       0.0    14194.672424
                    1.0    14237.784236
                    2.0    14162.106413
                    3.0    14178.602777
Name: cost_final, dtype: float64

Corsa completa (protocollo articolo: R ∈ {1,10,100}, ℓ/k ∈ {0.1,…,10},
r = 0..15, mediana su 11 run, k=50). Attenzione: ore di CPU locali.

In [4]:
if RUN_FIG52_FULL:
    df52 = run_fig52(client=None, seed=SEED, n_runs=N_RUNS)
    _ts = time.strftime("%Y%m%d_%H%M%S")
    _csv = os.path.join(RESULTS_DIR, f"fig52_{_ts}.csv")
    df52.to_csv(_csv, index=False)
    print("Saved", _csv)
    plot_fig52(df52, output_dir="figures")
else:
    print("RUN_FIG52_FULL = False")

RUN_FIG52_FULL = False


## 3. Dati KDD **[CLUSTER]** — caricamento (come in analysis.ipynb)

Paths identici alle VM (NON modificare, vedi AGENTS.md). Per il 10% usare
`DATASET_URL_10PC`; per le tabelle usare `DATASET_URL_FULL`.

In [5]:
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = "/tmp/kddcup_data.parquet"

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label",
]

### Cluster on/off

In [6]:
# DO NOT RUN if already existing!
if RUN_CLUSTER:
    N_WORKERS = 8
    NUM_PARTITIONS = 8 * N_WORKERS
    cluster, client = launch_cluster(N_WORKERS)
else:
    print("RUN_CLUSTER = False")

Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-08-29 21:53:11,409 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:11,408 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-29 21:53:11,437 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:11,437 - distributed.scheduler - INFO - State start
2026-08-29 21:53:11,440 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:11,440 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-08-29 21:53:13,500 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:13,507 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.254:37475'
2026-08-29 21:53:13,525 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:13,531 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.145:33429'
2026-08-29 21:53:13,588 - distributed.deploy.ssh - INFO - 2026-08-29 21:53:13,596 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.18

Cluster avviato e connessione stabilita con successo!



## 4. Fig 5.1 — KDD 10%, exact-ℓ **[CLUSTER]**

Protocollo: k ∈ {17,33,65,129}, ℓ/k ∈ {1,2,4}, r = 1..10, mediana su 11 run.
Costo moderato: ~4·3·10·11 = 1320 run di seeding+Lloyd's sul 10%.

In [ ]:
if RUN_CLUSTER and RUN_FIG51:
    X_bag, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH,
                            PARQUET_PATH, COL_NAMES,
                            n_partitions=NUM_PARTITIONS, client=client)
    df51 = run_fig51(client, X_bag, seed=SEED, n_runs=N_RUNS,
                     num_partitions=NUM_PARTITIONS)
    df51.to_csv(os.path.join(RESULTS_DIR, "fig51_full.csv"), index=False)
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

Using cached dataset: /home/ubuntu/Project/libero_development/data/kddcup_data.gz
Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


/home/ubuntu/pyvenv/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (13) found smaller than n_clusters (17). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:729: UserWarning: 4 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig51] k=17, l=17 (l/k=1), r=1: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=2: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=3: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=4: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=5: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=6: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=7: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=8: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=9: 11 run completate


## 5. Table 3/4 — KDD full **[CLUSTER, nottata]**

Prima il sanity check (rischio #3 di ANALYSIS_PLAN: pool candidati grandi con
ℓ=10k), poi la sweep completa. Protocollo critico: `policy="fixed", r=5`
(gestito dentro `run_table34`) — la regola auto ℓ/k≤0.1→15 NON è quella
della tabella.

In [ ]:
if RUN_CLUSTER and RUN_TABLE_SANITY:
    X_bag_full, _ = load_dataset(DATASET_URL_FULL, RAW_GZ_PATH, PARQUET_PATH,
                                 PARQUET_PATH, COL_NAMES,
                                 n_partitions=NUM_PARTITIONS, client=client)
    # singola run piu' pesante: k=500, l=5000 -> pool atteso ~25k candidati
    from src.paper_experiments import _run_one_parallel
    import time as _t
    _t0 = _t.time()
    res = _run_one_parallel(X_bag_full, k=500, l=5000, r=5, run_seed=SEED,
                            policy="fixed", max_iter_fit=10)
    print(res)
    print(f"totale {_t.time()-_t0:.1f}s")
else:
    print("RUN_CLUSTER/RUN_TABLE_SANITY = False")

In [ ]:
if RUN_CLUSTER and RUN_TABLE34:
    df34 = run_table34(client, X_bag_full, seed=SEED, n_runs=N_RUNS,
                       num_partitions=NUM_PARTITIONS)
    df34.to_csv(os.path.join(RESULTS_DIR, "table34_full.csv"), index=False)
else:
    print("RUN_CLUSTER/RUN_TABLE34 = False")

## 6. Analisi: figure e tabelle stile-articolo

Dai CSV salvati (funziona anche su questa macchina dopo aver riportato i CSV
in `results/`, oppure direttamente dai DataFrame delle sezioni sopra).

In [ ]:
# Esempio (decommentare quando i CSV esistono):
# p51 = os.path.join(RESULTS_DIR, "fig51_full.csv")
# p52 = os.path.join(RESULTS_DIR, "fig52_<timestamp>.csv")
# p34 = os.path.join(RESULTS_DIR, "table34_full.csv")
# plot_fig51(p51, output_dir="figures")
# plot_fig52(p52, output_dir="figures")
# display(table34_cost_table(p34))     # Table 3, costi x1e-10 (mediane)
# display(table34_time_table(p34))     # Table 4, tempi (mediane)

## 7. Confronto con i valori del paper

Compilare dopo ogni artefatto: tabella obtained-vs-paper (costi ×10⁻¹⁰,
mediane). Valori di riferimento nel PDF: `docs/1203.6402v1.pdf`,
Tabelle 3/4 e Figures 5.1/5.2.

| Artefatto | Configurazione | Paper | Ottenuto | Note |
|---|---|---|---|---|
| Table 3 | k=500, ℓ/k=1, r=5, final | *(da PDF)* | | |
| ... | | | | |

## 8. Spegnimento cluster

In [ ]:
# Da eseguire a fine lavoro
shutdown_cluster(cluster, client)